In [74]:
import pandas as pd

#### Step 1. Data Preparation

Load Datasets

**Data Desciption**

Energy Performance Certificates (EPC) dataset: Domestic Energy Performance Certificates subset including January to July 2026 obtained through [Link](https://get-energy-performance-data.communities.gov.uk/).


In [75]:
ppd_columns = [
    'transaction unique identifier',
    'price',
    'date of transfer',
    'postcode',
    'property type',
    'old/new',
    'duration',
    'PAON',
    'SAON',
    'street',
    'locality',
    'town/city',
    'district',
    'county',
    'PPD category type',
    'record status'
]

In [76]:
hmlr = pd.read_csv("data/pp-2026.csv", header=None, names=ppd_columns)
epc = pd.read_csv("data/epc-2026.csv")

Data Cleaning

① Normalize postcode without space

In [77]:
def clean_postcode(series):
    return (
        series
        .fillna("")
        .astype(str)
        .str.upper()
        .str.replace(r"\s+", "", regex=True)
        .str.strip()
    )

In [78]:
hmlr["postcode_clean"] = clean_postcode(hmlr["postcode"])
epc["postcode_clean"] = clean_postcode(epc["postcode"])


② Normalize address with uppercase and no comma

In [79]:
epc["epc_address"] = epc["address"].str.upper().str.strip()

# Combine segments into full address in HMLR
hmlr["hmlr_address"] = (
    hmlr["SAON"].fillna("").astype(str).str.strip()
    + " "
    + hmlr["PAON"].fillna("").astype(str).str.strip()
    + " "
    + hmlr["street"].fillna("").astype(str).str.strip()
).str.replace(r"\s+", " ", regex=True).str.strip()

hmlr["hmlr_address"] = hmlr["hmlr_address"].str.replace(",", "", regex=False)
epc["epc_address"] = epc["epc_address"].str.replace(",", "", regex=False)

print(hmlr[["PAON", "SAON", "street", "hmlr_address"]].iloc[29:40].to_string(index=True))

                  PAON     SAON            street                                hmlr_address
29                  95   FLAT 8    CHALFONT DRIVE                    FLAT 8 95 CHALFONT DRIVE
30                  59      NaN     AUSTIN STREET                            59 AUSTIN STREET
31                  80      NaN    CHALFONT DRIVE                           80 CHALFONT DRIVE
32  GALLIARD COURT, 48  FLAT 59  BARONSON GARDENS  FLAT 59 GALLIARD COURT 48 BARONSON GARDENS
33                  18      NaN      GRENDON WALK                             18 GRENDON WALK
34       OAKTREE COURT  FLAT 28     GEORGE STREET         FLAT 28 OAKTREE COURT GEORGE STREET
35                  13      NaN    FARMCLOSE ROAD                           13 FARMCLOSE ROAD
36                   4      NaN      NIBBITS LANE                              4 NIBBITS LANE
37                  17      NaN   BEECHWOOD DRIVE                          17 BEECHWOOD DRIVE
38                 112      NaN        HALSE ROAD           

③ Normalize EPC construction age without prefix

In [80]:
epc["epc_construction_age_band"] = (
    epc["construction_age_band"].astype(str).str.strip().str.replace("England and Wales: ", "", regex=False).str.replace("before ", "", regex=False)
)

print(epc[["epc_construction_age_band", "epc_address"]].iloc[29:40].to_string(index=True))

   epc_construction_age_band                             epc_address
29                      1900             THE MANOR CLIFFORD CHAMBERS
30                      1900       FLAT A 11 PARK ROAD COLLIERS WOOD
31                 1991-1995                         5 YEW TREE LANE
32                      2022        40 DRIVER CRESCENT LITTLE HULTON
33                 1900-1929       FLAT 7 ST. MARTINS 9 CLANDON ROAD
34                 1900-1929                  119B WHIPPS CROSS ROAD
35                      1900  FLAT 6 WESTFIELD HOUSE MANSBRIDGE ROAD
36                 1976-1982                FLAT 2 33 CHALGROVE ROAD
37                 1900-1929                        232 STAINES ROAD
38                 2012-2021   26 ENDURANCE HOUSE 9-11 GENEVA STREET
39                 1930-1949                137 WESTCLIFF PARK DRIVE


④ Normalize HMLR transaction date without time

In [81]:
hmlr["hmlr_date_of_transfer"] = pd.to_datetime(hmlr["date of transfer"],errors="coerce").dt.strftime("%Y-%m-%d")

print(hmlr[["hmlr_date_of_transfer", "hmlr_address"]].iloc[29:40].to_string(index=True))

   hmlr_date_of_transfer                                hmlr_address
29            2026-05-29                    FLAT 8 95 CHALFONT DRIVE
30            2026-01-26                            59 AUSTIN STREET
31            2026-06-26                           80 CHALFONT DRIVE
32            2026-05-12  FLAT 59 GALLIARD COURT 48 BARONSON GARDENS
33            2026-04-08                             18 GRENDON WALK
34            2026-06-01         FLAT 28 OAKTREE COURT GEORGE STREET
35            2026-06-10                           13 FARMCLOSE ROAD
36            2026-01-16                              4 NIBBITS LANE
37            2026-06-11                          17 BEECHWOOD DRIVE
38            2026-05-29                              112 HALSE ROAD
39            2026-01-26                            33 LYNTON AVENUE


Data Matching

Match property with the same postcode followed by the same full address

In [82]:
postcode_matched = hmlr.merge(
    epc,
    on="postcode_clean",
    how="inner",
    suffixes=("_hmlr", "_epc")
)

print("\nMatched rows:", len(postcode_matched))
print("Matched HMLR properties:", postcode_matched["transaction unique identifier"].nunique())



Matched rows: 287663
Matched HMLR properties: 139047


In [84]:
# Examples of matched postcode

display(
    postcode_matched[
        [   
            "hmlr_address",
            "epc_address",
            "postcode_clean"
        ]
    ].head(40)
)

,hmlr_address,epc_address,postcode_clean
0,153 HOLLYGATE LANE,151 HOLLYGATE LANE COTGRAVE,NG123UJ
1,7 FIRST AVENUE,6 FIRST AVENUE RAINWORTH,NG210BU
2,7 FIRST AVENUE,7 FIRST AVENUE RAINWORTH,NG210BU
3,25 EREWASH GROVE,32 EREWASH GROVE TOTON,NG96EY
4,25 EREWASH GROVE,40 EREWASH GROVE TOTON,NG96EY
5,24 PENARTH GARDENS,37 PENARTH GARDENS,NG54EG
6,24 PENARTH GARDENS,24 PENARTH GARDENS,NG54EG
7,25 REDGATE PLACE,19 REDGATE PLACE EAST LEAKE,LE126AA
8,25 REDGATE PLACE,23 REDGATE PLACE EAST LEAKE,LE126AA
9,25 REDGATE PLACE,16 REDGATE PLACE EAST LEAKE,LE126AA


In [85]:
address_matched = postcode_matched[
    postcode_matched["epc_address"] == postcode_matched["hmlr_address"]
].copy()

print("\nAddress matched rows:", len(address_matched))
print(
    "Address matched HMLR properties:",
    address_matched["transaction unique identifier"].nunique()
)


Address matched rows: 13268
Address matched HMLR properties: 13071


In [86]:
# Inspect duplicate matches: some properties have multiple EPC inspections.
# Preserve all EPC records as separate records rather than treating them as duplicates.

duplicate_hmlr = (
    address_matched[
        address_matched["transaction unique identifier"].duplicated(
            keep=False
        )
    ]
    .sort_values("transaction unique identifier")
)
print(
    duplicate_hmlr[
        ["transaction unique identifier", "postcode_clean", "hmlr_address", "epc_address", "hmlr_date_of_transfer",  "inspection_date", "certificate_number",]
    ].head(10).to_string(index=False)
)


         transaction unique identifier postcode_clean              hmlr_address               epc_address hmlr_date_of_transfer inspection_date       certificate_number
{49E87C31-8DF6-591C-E063-4704A8C00C31}        MK402TR FLAT D 51 DE PARYS AVENUE FLAT D 51 DE PARYS AVENUE            2026-01-15      2026-01-09 3736-4929-3500-0741-7202
{49E87C31-8DF6-591C-E063-4704A8C00C31}        MK402TR FLAT D 51 DE PARYS AVENUE FLAT D 51 DE PARYS AVENUE            2026-01-15      2026-01-14 3700-0943-0722-7594-3963
{49E87C32-0107-591C-E063-4704A8C00C31}        PO301RF            4 WINSTON ROAD            4 WINSTON ROAD            2026-01-05      2026-06-09 2176-1183-9385-7212-1114
{49E87C32-0107-591C-E063-4704A8C00C31}        PO301RF            4 WINSTON ROAD            4 WINSTON ROAD            2026-01-05      2026-01-27 2901-1183-5567-6408-1071
{49E87C32-4BA9-591C-E063-4704A8C00C31}        OX118RE              85 SAMOR WAY              85 SAMOR WAY            2026-01-08      2026-04-17 9433-3063-7

In [87]:
# Examples of matched address

display(
    address_matched[
        [   
            "hmlr_address",
            "epc_address",
            "postcode_clean"
        ]
    ].head(10)
)

,hmlr_address,epc_address,postcode_clean
6,24 PENARTH GARDENS,24 PENARTH GARDENS,NG54EG
25,12 HILLCREST MEWS,12 HILLCREST MEWS,DN226RB
32,FLAT 59 GALLIARD COURT 48 BARONSON GARDENS,FLAT 59 GALLIARD COURT 48 BARONSON GARDENS,NN14NU
45,10 CENTRAL AVENUE,10 CENTRAL AVENUE,NN28DZ
46,40 EAST PADDOCK COURT,40 EAST PADDOCK COURT,NN38LF
66,44 PENNINE WAY,44 PENNINE WAY,NN169AX
93,51 CITRON ROAD,51 CITRON ROAD,NN81UN
110,23 MEADOWVALE CRESCENT,23 MEADOWVALE CRESCENT,NG119LY
128,5 SCHOOL CLOSE,5 SCHOOL CLOSE,NG22GG
168,47 RICHMOND ROAD,47 RICHMOND ROAD,NN126EX


In [88]:
# Store relevant information seperately
linked_properties = address_matched[
    ["transaction unique identifier", "postcode_clean", "hmlr_address", "property type", "property_type", "built_form", "price", "hmlr_date_of_transfer", "inspection_date", "transaction_type", "old/new", "epc_construction_age_band", "floor_level", "total_floor_area", "number_habitable_rooms"]
].copy()

linked_properties.columns = ["transaction_id", "postcode", "address", "HMLR_property_type", "EPC_property_type", "EPC_built_form", "HMLR_price", "HMLR_transaction_date", "EPC_inspection_date", "EPC_transaction_type", "HMLR_old/new", "EPC_construction_age_band", "EPC_floor_level", "EPC_total_floor_area", "EPC_number_habitable_rooms"]

In [89]:
linked_properties.to_csv("data/linked_properties.csv",index=False)

#### Step 2. Observe Conflicts

Preliminary inspection of the linked data to identify potential conflict fields.
Full conflict detection is implemented in *find_conflicts.py*.

① "built_form" in EPC **VS** "property type" in HMLR

In [90]:
display(
    linked_properties[
        [   
            "postcode",
            "HMLR_property_type",
            "EPC_built_form"
        ]
    ].head(30)
)

,postcode,HMLR_property_type,EPC_built_form
6,NG54EG,T,Semi-Detached
25,DN226RB,S,Mid-Terrace
32,NN14NU,F,Not Recorded
45,NN28DZ,F,Semi-Detached
46,NN38LF,T,End-Terrace
66,NN169AX,S,Semi-Detached
93,NN81UN,S,Semi-Detached
110,NG119LY,S,Semi-Detached
128,NG22GG,T,Semi-Detached
168,NN126EX,T,Mid-Terrace


② "property_type" in EPC **VS** "property type" in HMLR

In [91]:
display(
    linked_properties[
        [   
            "postcode",
            "HMLR_property_type",
            "EPC_property_type"
        ]
    ].head(30)
)

,postcode,HMLR_property_type,EPC_property_type
6,NG54EG,T,House
25,DN226RB,S,House
32,NN14NU,F,Flat
45,NN28DZ,F,Bungalow
46,NN38LF,T,House
66,NN169AX,S,Bungalow
93,NN81UN,S,House
110,NG119LY,S,House
128,NG22GG,T,Bungalow
168,NN126EX,T,House


③ "old/new" in EPC **VS** "construction_age_band" in HMLR

In [92]:
display(
    linked_properties[
        [   
            "postcode",
            "HMLR_transaction_date", 
            "EPC_inspection_date",
            "HMLR_old/new",
            "EPC_construction_age_band"
        ]
    ].iloc[269:290]
)

,postcode,HMLR_transaction_date,EPC_inspection_date,HMLR_old/new,EPC_construction_age_band
4272,CM111BL,2026-05-19,2026-01-21,N,1967-1975
4284,SS69SB,2026-06-12,2026-01-19,N,1983-1990
4286,RM176PW,2026-06-02,2026-02-01,N,2003-2006
4319,SS131AW,2026-05-29,2026-02-16,N,1996-2002
4323,CM40PQ,2026-05-28,2026-05-25,N,1950-1966
4328,CF390LU,2026-04-30,2026-02-25,N,1900
4360,CV64HQ,2026-05-14,2026-01-05,N,1967-1975
4364,CV346NY,2026-05-05,2026-01-21,N,2012-2021
4366,B735LF,2026-04-23,2026-01-29,N,1967-1975
4369,SR29AT,2026-04-27,2026-05-06,N,2023


④ Examine the relationship between EPC floor area / room count and property type, as a basis for decision rules

In [93]:
house_group = linked_properties[linked_properties["EPC_property_type"].isin(["House", "Bungalow"])]

print(house_group["EPC_total_floor_area"].quantile([0.05, 0.10, 0.50, 0.75, 0.90, 0.95]))
print(house_group["EPC_number_habitable_rooms"].quantile([0.05, 0.10, 0.50, 0.75, 0.90, 0.95]))

0.05     56.0
0.10     61.0
0.50     85.0
0.75    104.0
0.90    132.0
0.95    153.0
Name: EPC_total_floor_area, dtype: float64
0.05    3.0
0.10    3.0
0.50    5.0
0.75    5.0
0.90    6.0
0.95    7.0
Name: EPC_number_habitable_rooms, dtype: float64


In [94]:
flat_group = linked_properties[linked_properties["EPC_property_type"].isin(["Flat", "Maisonette"])]

print(flat_group["EPC_total_floor_area"].quantile([0.05, 0.10, 0.50, 0.75, 0.90, 0.95]))
print(flat_group["EPC_number_habitable_rooms"].quantile([0.05, 0.10, 0.50, 0.75, 0.90, 0.95]))

0.05     36.0
0.10     40.0
0.50     59.0
0.75     71.0
0.90     86.0
0.95    100.0
Name: EPC_total_floor_area, dtype: float64
0.05    2.0
0.10    2.0
0.50    3.0
0.75    3.0
0.90    4.0
0.95    4.0
Name: EPC_number_habitable_rooms, dtype: float64
